# Reflex-2 quickstart

Judge home-camera video locally: fall detection with the released head, a live-camera style timeline, and your own head trained in seconds.

Runtime: a GPU runtime (T4 or better) is recommended. The first run downloads google/embeddinggemma-2 (1.5 GB).

In [ ]:
!git clone -q https://github.com/matu79go/reflex-2
%cd reflex-2
!pip -q install -r requirements.txt

Sample clips from GMDCSA24 (MIT License): staged falls and daily activities in real homes.

In [ ]:
import os, urllib.request, urllib.parse
RAW = "https://raw.githubusercontent.com/ekramalam/GMDCSA24-A-Dataset-for-Human-Fall-Detection-in-Videos/master"
def get(path):
    dst = path.replace("/", "_").replace(" ", "")
    if not os.path.exists(dst):
        urllib.request.urlretrieve(RAW + "/" + urllib.parse.quote(path), dst)
    return dst
fall_clip = get("Subject 2/Fall/02.mp4")
daily_clip = get("Subject 1/ADL/01.mp4")


In [ ]:
from reflex2 import Reflex2
rf = Reflex2().load_heads("heads")
print(rf.device)
print(rf.decide(fall_clip, "fall"))
print(rf.decide(daily_clip, "fall"))

## Watch like a live camera

Every 0.5 s, judge the last 4 s.

In [ ]:
import matplotlib.pyplot as plt
tl = rf.watch(fall_clip, "fall")
t, p = [x[0] for x in tl], [x[1]["fall"] for x in tl]
plt.figure(figsize=(8, 3)); plt.plot(t, p, marker="o"); plt.axhline(0.5, ls="--", c="gray")
plt.ylim(0, 1); plt.xlabel("seconds"); plt.ylabel("P(fall)"); plt.title("Fall probability over time"); plt.show()

## Your own head from a few examples

6 falls and 6 daily activities from three people; the clips above (another person) are the test. Real use: a few dozen clips from your own cameras.

In [ ]:
ppl = [(1, (1, 2)), (3, (1, 2)), (4, (1, 3))]
fall = [get(f"Subject {s}/Fall/{i:02d}.mp4") for s, ids in ppl for i in ids]
daily = [get(f"Subject {s}/ADL/{i:02d}.mp4") for s, ids in ppl for i in (2, 3)]
head = rf.train_head({"fall": fall, "daily": daily}, name="my_fall")
print(rf.decide(fall_clip, "my_fall"))
print(rf.decide(daily_clip, "my_fall"))
head.save("heads/my_fall.json")

## Untrained question

Similarity to short descriptions; good for 'what is this', weaker for judgments.

In [ ]:
print(rf.zero_shot(daily_clip, {"bedroom": "a bedroom", "kitchen": "a kitchen", "office": "an office"}))